# Complex Helmholtz MHM: a resolved plane wave and variational workflow

Define outgoing physical data, local Helmholtz forms, a normal-flux interface
and global pressure continuity by hand. Use generic mesh, form, assembly and
solve abstractions; the refinement plots refer to this same declared family.
The locked `introduction` environment executes the native UFL calculations.

## 1. Derive the wave and its boundary data

On the unit square with unit density and modulus, choose $\omega=10\pi$,
$d=(\cos(\pi/13),\sin(\pi/13))$ and $p_*(x)=\exp(i\omega d\cdot x)$.
The volume source is zero because $\lVert d\rVert=1$:

$$
\begin{aligned}
-\Delta p-\omega^2p&=0,\\
\partial_np-i\omega p&=g,
&g&=i\omega(d\cdot n-1)p_* \quad\text{on }\partial\Omega.
\end{aligned}
$$

Represent complex pressure by two interleaved real components $z=(\Re p,\Im p)$.
The two components are algebraic coordinates, rather than a physical vector.
The weak impedance contribution is $\omega(z_1v_0-z_0v_1)$. This exact real/imaginary
embedding keeps the outgoing sign. Exterior data act only on the physical
boundary; local interior faces still carry the unknown negative normal gradient.


In [ ]:
from pathlib import Path
import os
import sys
from pymhm.io.workspace import workspace_from_archive

# Download verified support files; this operation does not execute them.
COMPANION_URL = "https://ipes-lncc.github.io/pymhm/downloads/577a8c2a43795777c48ec7471ece35c6365c63ceeb5d2b293712de08f990d5d0/introductory_methods-companion.zip"
COMPANION_SHA256 = "577a8c2a43795777c48ec7471ece35c6365c63ceeb5d2b293712de08f990d5d0"
WORKSPACE = Path(
    os.environ.get("PYMHM_WORKSPACE", Path.cwd() / ".pymhm-companions" / COMPANION_SHA256)
)
ROOT = workspace_from_archive(COMPANION_URL, sha256=COMPANION_SHA256, directory=WORKSPACE)
os.environ["PYMHM_WORKSPACE"] = str(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Explicitly prepare declared data with the downloaded Python helpers.
from scripts.notebook_reproduction import notebook_workspace

ROOT = notebook_workspace("waves/helmholtz/introductory_methods.ipynb", directory=ROOT)
root = ROOT
print("Workspace:", ROOT)

import numpy as np


## 2. Write the UFL local forms and trace pairings

Choose continuous local Q4 pressure on a $2\times2$ subdivision of every
macro square, and vector P2 coordinates for the complex normal-flux trace.
The definition cell below writes the common volume and impedance form, source,
local trial pairing and independent global test pairing:

$$
\begin{aligned}
 a_K(p,v)&=(\nabla p,\nabla v)_K-\omega^2(p,v)_K
                   -\langle i\omega p,v\rangle_{\partial K\cap\partial\Omega},\\
 b_K(\lambda,v)&=\langle\lambda,v\rangle_{\partial K\setminus\partial\Omega},\\
 c_K(p,\mu)&=-\langle p,\mu\rangle_{\partial K\setminus\partial\Omega}.
\end{aligned}
$$

`local.native_space` owns native coefficient ordering. `local.trace_pairings`
owns geometric incidence signs. Declare the mathematical signs directly;
do not implement face orientations in the notebook. There are no retained
static modes at the selected nonresonant frequency. Positive frequency alone
does not establish local invertibility at arbitrary frequencies.


In [ ]:
import numpy as np
import ufl
import basix
import basix.ufl
from types import SimpleNamespace
from pymhm import (Equation, TriangleMesh, FaceSpace, SkeletonSpace, MeshHierarchy,
                   LocalContext, bind_interface, bind_problem, assemble, solve,
                   MultiscaleProblem, compile_form)
from pymhm.meshes.cartesian import CartesianMacroMesh
from pymhm.backends.spaces import bind_space, coefficient_map
from pymhm.fem.scalar.quadrilateral import qk_space, quadrilateral_quadrature
from examples.minimal_flow_originals import original_diagnostics

wave_frequency=10*np.pi
wave_direction=np.array((np.cos(np.pi/13),np.sin(np.pi/13)))

def wave_truth(points):
    """Evaluate the independently specified unit-amplitude plane wave as Re/Im."""
    phase=wave_frequency*(np.asarray(points)@wave_direction)
    return np.column_stack((np.cos(phase),np.sin(phase)))

def wave_gradient(points):
    """Differentiate the analytical wave in physical Cartesian coordinates."""
    phase=wave_frequency*(np.asarray(points)@wave_direction)
    return wave_frequency*np.stack((-np.sin(phase),np.cos(phase)),axis=1)[:,:,None]*wave_direction

def wave_forms(V,domain):
    """Write the Helmholtz volume and outgoing physical-boundary impedance forms."""
    z,v=ufl.TrialFunction(V),ufl.TestFunction(V)
    x=ufl.SpatialCoordinate(domain)
    n=ufl.FacetNormal(domain)
    exterior=ufl.conditional(ufl.Or(ufl.Or(ufl.lt(x[0],1e-12),ufl.gt(x[0],1-1e-12)),
                                  ufl.Or(ufl.lt(x[1],1e-12),ufl.gt(x[1],1-1e-12))),1.,0.)
    phase=wave_frequency*(x[0]*wave_direction[0]+x[1]*wave_direction[1])
    normal_factor=wave_frequency*(ufl.dot(ufl.as_vector(wave_direction),n)-1)
    boundary_source=ufl.as_vector((-normal_factor*ufl.sin(phase),normal_factor*ufl.cos(phase)))
    dx=ufl.Measure('dx',domain=domain,metadata={'quadrature_degree':16})
    ds=ufl.Measure('ds',domain=domain,metadata={'quadrature_degree':16})
    a=(ufl.inner(ufl.grad(z),ufl.grad(v))-wave_frequency**2*ufl.inner(z,v))*dx
    a+=wave_frequency*(z[1]*v[0]-z[0]*v[1])*exterior*ds
    L=ufl.inner(boundary_source,v)*exterior*ds
    return a,L,exterior

wave_element=basix.ufl.element('Lagrange','quadrilateral',4,shape=(2,),
                               lagrange_variant=basix.LagrangeVariant.equispaced)

def wave_local(local: LocalContext):
    """Declare Q4 complex pressure, P2 normal moments and outgoing exterior data."""
    native=local.native_space(wave_element)
    z,v=ufl.TrialFunction(native.space),ufl.TestFunction(native.space)
    a,L,exterior=wave_forms(native.space,native.mesh)
    b=local.trace_pairings(lambda phi,ds:ufl.inner(phi,v)*(1-exterior)*ds)
    c=local.trace_pairings(lambda phi,ds:-ufl.inner(phi,z)*(1-exterior)*ds,axis='rows')
    local.field('pressure',native)
    return local.equations(a=a,L=L,b=b,c=c)

def wave_errors(fields,order):
    """Integrate physical relative pressure/gradient errors with independent quadrature."""
    reference,weights=quadrilateral_quadrature(order)
    squared=np.zeros(2)
    for field in fields:
        fine=field.mesh
        origins=fine.points[fine.cells[:,0]]
        points=(origins[:,None,:]+reference[None,:,:]*fine.spacing).reshape(-1,2)
        cell_ids=np.repeat(np.arange(len(fine.cells)),len(reference))
        values,gradient=field.values_and_gradient(points,cells=cell_ids)
        physical_weights=(fine.areas[:,None]*weights[None,:]).ravel()
        squared[0]+=np.sum(physical_weights*np.sum((values-wave_truth(points))**2,axis=1))
        squared[1]+=np.sum(physical_weights*np.sum((gradient-wave_gradient(points))**2,axis=(1,2)))
    return {'pressure_relative_l2':float(np.sqrt(squared[0])),
            'gradient_relative_l2':float(np.sqrt(squared[1])/wave_frequency)}



## 3. Define the meshes and global equation, assemble and solve

The multiplier is the negative physical normal gradient on interior faces.
Exterior multiplier coordinates are zero placeholders because the Robin data
are already in the local impedance form and load. They are not an assertion
that the physical exterior flux vanishes. Fix only these placeholders to zero.
The global equation sums incident pressure moments with the negative sign
chosen in $c_K$; its right-hand side is zero.

After `assemble` and `solve`, evaluate the named pressure fields using their
executed bases. Independent quadrature orders 10 and 14 integrate physical
relative pressure and broken gradient errors. The separate original-row
checker measures uncondensed coefficient equations, not a physical field norm.


In [ ]:
wave_macro=CartesianMacroMesh(12)
wave_skeleton=SkeletonSpace(wave_macro,tuple(FaceSpace.uniform(2) for _ in wave_macro.faces),components=2)
wave_hierarchy=MeshHierarchy(wave_macro,tuple(wave_macro.submesh(i,2) for i in range(len(wave_macro.cells))))
wave_interface=bind_interface(wave_skeleton,convention='normal')
wave_problem=bind_problem(wave_hierarchy,wave_interface,wave_local,global_equation=Equation(0,0))
wave_system=assemble(wave_problem)
wave_fixed={int(dof):0. for face in wave_macro.boundary_faces for dof in wave_skeleton.dofs(int(face))}
wave_solution=solve(wave_system,fixed=wave_fixed)
wave_pressure=wave_solution.field('pressure')
wave_norms=wave_errors(wave_pressure,10)
wave_norms_finer=wave_errors(wave_pressure,14)
wave_original=original_diagnostics(SimpleNamespace(hybrid=wave_solution),
    SimpleNamespace(system=wave_system,applied_boundary=-wave_system.global_load[:wave_system.trace_size],
                    fixed=wave_fixed,mean_weights=[],mean_values=[],constraints=[]),
    [{'complex_pressure_native_block':len(values)} for values in wave_solution.fields])
assert wave_original['accepted']
assert all(np.isclose(wave_norms[k],wave_norms_finer[k],rtol=1e-7,atol=0) for k in wave_norms)
print('By-hand UFL plane-wave errors:',wave_norms,flush=True)
print('Original local/global rows:',{k:v for k,v in wave_original.items() if k!='blocks'},flush=True)



## 4. Independently refine a classical Galerkin reference

Use the same `wave_forms` on one globally conforming Q4 mesh, with the same
outgoing physical data and no MHM condensation. `MultiscaleProblem.from_global`
allows the same general assembly/solve workflow. The native adapter closes its
resources explicitly; the portable coefficient map and declared field preserve
the executed nodal basis for evaluation.

Refine the reference on global grids 16, 32 and 64 before plotting its finest
field. Both analytical norms must decrease; a fine reference remains numerical.


In [ ]:
def classical_wave(n):
    """Independently assemble conforming Q4 Galerkin with the same outgoing UFL data."""
    fine=CartesianMacroMesh(n)
    native=bind_space(fine,element=wave_element)
    try:
        a,L,_=wave_forms(native.space,native.mesh)
        problem=MultiscaleProblem.from_global(Equation(a,L),native.size)
        system=assemble(problem)
        state=solve(system)
        _,points=qk_space(fine,4)
        canonical=state.trace[coefficient_map(native)]
    finally:
        native.close()
    from pymhm.postprocessing.nodal import nodal_field
    from pymhm.postprocessing.fields import DiscreteField
    field=DiscreteField(nodal_field('pressure',fine,4,components=2),canonical)
    errors=wave_errors((field,),10)
    errors_finer=wave_errors((field,),14)
    assert all(np.isclose(errors[k],errors_finer[k],rtol=1e-7,atol=0) for k in errors)
    assert state.raw_residual<=1e-10
    return field,{'n':n,**errors,'global_compatibility_relative':state.raw_residual}

wave_references=[]
for n in (16,32,64):
    wave_reference,row=classical_wave(n)
    wave_references.append(row)
assert all(wave_references[i+1][name]<wave_references[i][name]
           for i in range(2) for name in wave_norms)
print('Independent global Q4 references:',wave_references,flush=True)



## 5. Plot the complex wave and its errors

Each macrocell is sampled independently. The actual macro mesh is drawn on
analytical, multiscale, reference and error panels. Individual colorbars remain
independent; reconstructed interface values are not averaged.


In [ ]:
import matplotlib.pyplot as plt
from examples.introduction.vector import plot_field_panels
from pathlib import Path
display=CartesianMacroMesh(8)
display_triangles=np.vstack((display.cells[:,(0,1,2)],display.cells[:,(0,2,3)]))
points=[];triangles=[];pressure=[];offset=0
for cell,field in enumerate(wave_pressure):
    vertices=wave_macro.points[wave_macro.cells[cell]]
    mapped=vertices[0]+display.points*(vertices[2]-vertices[0])
    points.append(mapped);triangles.append(display_triangles+offset)
    pressure.append(field.evaluate(mapped));offset+=len(mapped)
points=np.concatenate(points);triangles=np.concatenate(triangles);pressure=np.concatenate(pressure)
truth=wave_truth(points);reference=wave_reference.evaluate(points)
fig=plot_field_panels(wave_macro,{
    'Analytical Re pressure':(points,triangles,truth[:,0]),
    'MHM Re pressure':(points,triangles,pressure[:,0]),
    'Fine Galerkin Re pressure':(points,triangles,reference[:,0]),
    'Analytical Im pressure':(points,triangles,truth[:,1]),
    'MHM pressure error magnitude':(points,triangles,np.linalg.norm(pressure-truth,axis=1)),
    'Galerkin pressure error magnitude':(points,triangles,np.linalg.norm(reference-truth,axis=1)),
},figsize=(12,8))
output=Path('build/tutorial-methods');output.mkdir(parents=True,exist_ok=True)
fig.savefig(output/'helmholtz-wave-fields.png',dpi=200,
    metadata={'Author':'IPES Research Group','License':'CC BY 4.0 — https://creativecommons.org/licenses/by/4.0/'})
plt.show()


## 6. Reach the fixed-frequency asymptotic regime

The recorded family uses exactly the plane wave, Robin convention, square macro
meshes, local Q4/refinement two and polynomial P2 traces defined above. It
retains all $n=12,16,24,32,48,64$ observations. The native n12 solve above also
checks that its two physical norms agree with the attributed record.

For the smooth fixed-frequency regime, Chaumont-Frelet and Valentin (2020),
Section 6.3, Figures 6.6 and 6.7, report pressure order $\ell+2$ and broken
gradient order $\ell+1$ for polynomial traces. Here $\ell=2$: target orders are
four and three. The final three pressure orders are 3.9910, 3.9918 and 3.9930;
gradient orders are 2.9906, 2.9928 and 2.9941. The final four-level normalized
amplitudes vary by less than one percent.

The record distinguishes PyMHM analytical errors from values digitized from
the published figures. Local degrees and refinements are explicitly declared
here; equality with an unidentified historical local mesh is not claimed.
These rates are not a frequency-uniform pollution or heterogeneous resonance
estimate.


In [ ]:
import json
from examples.tutorial_convergence import refinement_series,plot_method_series,asymptotic_summary
record_path=ROOT/'examples/results/helmholtz-article/published-convergence.json'
record=json.loads(record_path.read_text())
selected=[row for row in record['rows'] if row['ell']==2 and row['basis']=='polynomial']
rows=[]
for resolution in sorted({row['n'] for row in selected}):
    fields={row['field']:row['pymhm_relative_error'] for row in selected if row['n']==resolution}
    rows.append({'n':resolution,'H':1/resolution,**fields})
first=next(row for row in rows if row['n']==12)
for name,field in [('pressure_relative_l2','pressure'),('gradient_relative_l2','gradient')]:
    np.testing.assert_allclose(wave_norms[name],first[field],rtol=1e-8,atol=0)
series=refinement_series(record_path,rows,'H',
    {'relative pressure L2':'pressure','relative broken gradient L2':'gradient'},
    {'relative pressure L2':4.0,'relative broken gradient L2':3.0},method='helmholtz',
    spaces='Q4/r2 local pressure; polynomial P2 complex normal traces; all exterior outgoing Robin data',
    rate_provenance='Chaumont-Frelet and Valentin (2020), Section6.3: fixed-frequency smooth pressure order four/gradient order three',
    refinement='macro grid spacing',root=ROOT)
print('Attributed input:',series['record'],series['sha256'])
print(asymptotic_summary(series))
figure=plot_method_series(series)
plt.show()


## Further control: an exactly represented affine field

The following low-frequency complex P2 patch preserves the earlier polynomial
reproduction example. Its geometric provider is a convenience adapter for
explicit volume and boundary forms. It has no meaningful fitted convergence
rate and is distinct from the resolved plane-wave experiment above.


## Declare and solve complex Helmholtz equations

The primary native forms use the exact real/imaginary embedding of complex
pressure; no complex coefficient is discarded. Unit density/modulus give
$-\Delta p-\omega^2p=f$ with $\omega=1/2$ and the independent affine
source $f=-\omega^2p$. This chosen finite patch has invertible local operators;
positive frequency alone does not exclude local resonances in other inputs.

$$
\begin{aligned}
a_T(p,v)&=(\nabla p,\nabla v)_T-\omega^2(p,v)_T,\\
B_T\lambda&=\langle\lambda,v\rangle_{\partial T},\\
C_Tp&=-\langle p,\mu\rangle_{\partial T}.
\end{aligned}
$$

The signed trace is $-\nabla p\cdot n$ in two interleaved real components.
The global RHS explicitly prescribes negative complex Dirichlet moments.
The native example has no absorption, PML or resonance-avoidance claim.


In [ ]:
import importlib.util
from pymhm import Equation, LocalEquations, MultiscaleProblem, assemble, columns, rows, compile_form
from pymhm.meshes.triangle import TriangleMesh
from pymhm.fem.traces.interval import FaceSpace, SkeletonSpace
from pymhm.fem.scalar.operators import boundary_data
from examples.variational_darcy import NativeFormProvider

native_available = importlib.util.find_spec("dolfinx") is not None
if native_available:
    import ufl
    from dolfinx import fem

    mesh = TriangleMesh.unit_square()
    trace_space = SkeletonSpace(mesh, components=2)
    omega = 0.5

    def exact_components(points):
        """Return real/imaginary components of (1+i)(1+x+2y)."""
        scalar = 1 + points[:, 0] + 2 * points[:, 1]
        return np.column_stack((scalar, scalar))

    def local_forms(context):
        """Write both real components of the complex Helmholtz equation."""
        space = fem.functionspace(context.space.mesh, ("Lagrange", 2, (2,)))
        u, v = ufl.TrialFunction(space), ufl.TestFunction(space)
        x = ufl.SpatialCoordinate(context.space.mesh)
        dx = ufl.dx(domain=context.space.mesh)
        exact = ufl.as_vector((1 + x[0] + 2 * x[1], 1 + x[0] + 2 * x[1]))
        return LocalEquations(
            (ufl.inner(ufl.grad(u), ufl.grad(v)) - omega**2 * ufl.inner(u, v)) * dx,
            ufl.inner(-(omega**2) * exact, v) * dx,
            columns(
                *(
                    float(sign) * v[j] * context.ds(side + 1)
                    for side, sign in enumerate(context.signs)
                    for j in range(2)
                )
            ),
            rows(
                *(
                    -float(sign) * u[j] * context.ds(side + 1)
                    for side, sign in enumerate(context.signs)
                    for j in range(2)
                )
            ),
            context.trace_dofs,
            metadata={"points": space.tabulate_dof_coordinates()[:, :2].copy()},
        )

    boundary, _ = boundary_data(trace_space, exact_components)
    problem = MultiscaleProblem(
        Equation(0, -boundary),
        NativeFormProvider(mesh, trace_space, local_forms, 2),
        range(len(mesh.cells)),
        trace_space.size,
        (0,) * len(mesh.cells),
    )
    system = assemble(problem)
    solution = system.solve()
    errors = [
        float(np.max(np.abs(field.reshape(-1, 2) - exact_components(record["points"]))))
        for field, record in zip(solution.fields, system.local_metadata, strict=True)
    ]
    assert max(errors) < 1e-10
    print(
        {
            "native_available": True,
            "spaces": "complex P2 pressure / P0 complex trace",
            "maximum_component_nodal_error_by_cell": errors,
            "global_original_relative_residual": solution.raw_residual,
        }
    )
else:
    print(
        {
            "native_available": False,
            "status": "User-written UFL primary solve not executed: select the Pixi fem kernel.",
        }
    )


## Compare the supported methods
These comparisons invoke the existing formulation implementations through the application catalogue and preserve their numerical records. The user-defined primary UFL solve above exercises the generic local/global API; the catalogue drivers remain compatibility comparisons.


In [ ]:
from examples.tutorial_scalar_variants import (
    VARIANTS,
    affine_pressure,
    affine_flux,
    run_variant,
)

record = run_variant("helmholtz")
print(record["spaces_and_conventions"])
print(record["metrics"])


## Plot pressure components and error with the macro mesh

Both components of this affine complex pressure are identical. Sampled nodal values reproduce the affine field; the panel triangulation displays those values separately in each macrocell. The macro mesh is drawn on every panel.


In [ ]:
assert native_available, "Select the locked introduction environment for this UFL demonstration."
import matplotlib.pyplot as plt
from matplotlib.tri import Triangulation

fig, axes = plt.subplots(2, 3, figsize=(10, 6), layout="constrained")
for component in range(2):
    for column in range(3):
        ax = axes[component, column]
        for values, record in zip(solution.fields, system.local_metadata, strict=True):
            points = record["points"]
            exact = exact_components(points)[:, component]
            numerical = values.reshape(-1, 2)[:, component]
            shown = (exact, numerical, np.abs(numerical - exact))[column]
            image = ax.tripcolor(Triangulation(points[:, 0], points[:, 1]), shown,
                                 shading="gouraud", vmin=0 if column==2 else 1,
                                 vmax=1e-10 if column==2 else 4)
        ax.triplot(mesh.points[:, 0], mesh.points[:, 1], mesh.cells, color="black", linewidth=0.6)
        name = "Real" if component==0 else "Imaginary"
        ax.set(title=f"{name}: {('exact', 'MHM', 'absolute error')[column]}",
               xlabel="x", ylabel="y", aspect="equal")
        fig.colorbar(image, ax=ax, label="Pressure" if column<2 else "Pressure error")
Path("build/tutorial-methods").mkdir(parents=True, exist_ok=True)
fig.savefig("build/tutorial-methods/helmholtz-affine-fields.png", dpi=200, metadata={"Author":"IPES Research Group", "License":"CC BY 4.0 — https://creativecommons.org/licenses/by/4.0/"})
plt.show()


## Independent conforming Galerkin reference

Assemble the same real/imaginary Helmholtz forms on global P2 grids of 16, 32 and 64 subdivisions. Prescribe the exact exterior pressure strongly and solve with the common sparse factorization owner. The affine field is exactly representable, so refinement verifies reproduction instead of supplying a fitted rate. The MHM example uses only two macrocells.


In [ ]:
from mpi4py import MPI
from dolfinx import mesh as native_mesh
from pymhm.linalg.linear import factorize

classical = []
for resolution in (16, 32, 64):
    domain = native_mesh.create_unit_square(MPI.COMM_SELF, resolution, resolution)
    space = fem.functionspace(domain, ("Lagrange", 2, (2,)))
    u, v = ufl.TrialFunction(space), ufl.TestFunction(space)
    x = ufl.SpatialCoordinate(domain)
    dx = ufl.dx(domain=domain)
    exact = ufl.as_vector((1+x[0]+2*x[1], 1+x[0]+2*x[1]))
    a = (ufl.inner(ufl.grad(u), ufl.grad(v)) - omega**2*ufl.inner(u,v))*dx
    L = -omega**2 * ufl.inner(exact, v)*dx
    points = space.tabulate_dof_coordinates()[:, :2]
    size = 2*len(points)
    matrix, load = compile_form(a, (size,size)), compile_form(L, (size,))
    exterior = np.any(np.isclose(points,0) | np.isclose(points,1), axis=1)
    fixed = (2*np.flatnonzero(exterior)[:,None] + np.arange(2)).ravel()
    free = np.setdiff1d(np.arange(size),fixed)
    values = np.zeros(size)
    values[fixed] = exact_components(points).ravel()[fixed]
    with factorize(matrix[free][:,free]) as factor:
        values[free] = factor.solve(load[free] - matrix[free][:,fixed] @ values[fixed])
    reference = fem.Function(space)
    reference.x.array[:] = values
    difference = reference-exact
    l2 = np.sqrt(fem.assemble_scalar(fem.form(ufl.inner(difference,difference)*dx)))
    assert l2 < 1e-10
    classical.append({"resolution":resolution,"pressure_L2":float(l2)})
classical


## References

- Chaumont-Frelet and Valentin (2020), *A Multiscale Hybrid-Mixed Method for the Helmholtz Equation in Heterogeneous Domains*. [DOI:10.1137/19M1255616](https://doi.org/10.1137/19M1255616).

The fixed-frequency analytical qualification declares its own local discretization;
it does not claim a matched historical table without its original inputs.
